# 06 多變項分析 — 參考解答

用松柏護理之家退伍軍人症 line list 練習 Modified Poisson regression（adjusted RR）
和邏輯斯迴歸（adjusted OR），並比較兩者差異。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (避免中文標籤顯示為方框) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- 讀取資料 ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## 題目 1：死亡預測 — Crude RR vs Crude OR

1. 建立 `dead` 欄位
2. 計算死亡率（case fatality rate）
3. 同時計算 crude RR（Modified Poisson）和 crude OR（logistic）
4. 整理成對照表格

In [ ]:
# --- 建立結果變項 ---
df["dead"] = (df["outcome"] == "dead").astype(int)

# 嚴重度轉數值（未感染者設為 0）
sev_map = {"not_ill": 0, "asymptomatic": 0, "mild": 1, "moderate": 2, "severe": 3}
df["severity_score"] = df["clinical_severity"].map(sev_map)

# 只用感染者做死亡預測（未感染者不會死於此疾病）
cases = df[df["infected"] == 1].copy()
cfr = cases["dead"].mean()
print(f"感染者：{len(cases)} 人，死亡：{cases['dead'].sum()} 人")
print(f"致死率 (CFR)：{cfr:.1%}")
print(f"→ CFR = {cfr:.1%}，比侵襲率 43% 低很多")
print(f"→ 預期 OR 和 RR 的差距會比感染預測時小\n")

# --- 同時計算 crude RR 和 crude OR ---
factors_death = ["age", "comorbidity_chf", "comorbidity_copd",
                 "immunosuppressed", "severity_score"]

crude_rows = []
for var in factors_death:
    # Modified Poisson → crude RR
    poisson = smf.glm(
        f"dead ~ {var}", data=cases,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0", disp=0)
    rr = np.exp(poisson.params[var])
    rr_ci = np.exp(poisson.conf_int().loc[var])

    # Logistic → crude OR
    logit = smf.logit(f"dead ~ {var}", data=cases).fit(disp=0)
    or_val = np.exp(logit.params[var])
    or_ci = np.exp(logit.conf_int().loc[var])

    crude_rows.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}\u2013{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}\u2013{or_ci[1]:.3f}",
    })

crude_df = pd.DataFrame(crude_rows)
print("=== 死亡預測：Crude RR vs Crude OR ===")
print(crude_df.to_string(index=False))
print("\n→ 死亡率較低（~16%），OR 和 RR 的差距比感染預測（43%）時小得多")

## 題目 2：多變項 Adjusted RR + Adjusted OR

建立預測死亡的多變項模型，同時用 Modified Poisson 和 Logistic Regression，
並排比較 adjusted RR 和 adjusted OR。

In [ ]:
# --- 共用公式 ---
formula_death = (
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score"
)

# --- Modified Poisson → Adjusted RR ---
poisson_multi = smf.glm(
    formula_death, data=cases,
    family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- Logistic → Adjusted OR ---
logit_multi = smf.logit(formula_death, data=cases).fit(disp=0, method="lbfgs")

# --- 並排比較表格 ---
compare_rows = []
for var in poisson_multi.params.index:
    if var == "Intercept":
        continue
    # Adjusted RR
    rr = np.exp(poisson_multi.params[var])
    rr_ci = np.exp(poisson_multi.conf_int().loc[var])
    # Adjusted OR
    or_val = np.exp(logit_multi.params[var])
    or_ci = np.exp(logit_multi.conf_int().loc[var])
    # OR 比 RR 高估多少
    pct_diff = (or_val - rr) / rr * 100

    compare_rows.append({
        "variable": var,
        "adj_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}\u2013{rr_ci[1]:.3f}",
        "adj_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}\u2013{or_ci[1]:.3f}",
        "OR高估%": f"{pct_diff:+.1f}%",
    })

compare_df = pd.DataFrame(compare_rows)
print("=== 死亡預測：Adjusted RR vs Adjusted OR ===")
print(compare_df.to_string(index=False))

# --- Crude vs Adjusted 比較 ---
print("\n=== Crude → Adjusted 變化（用 RR） ===")
for var in factors_death:
    c_row = crude_df[crude_df["variable"] == var].iloc[0]
    a_row = compare_df[compare_df["variable"] == var]
    if len(a_row) == 0:
        continue
    a_row = a_row.iloc[0]
    change = (a_row["adj_RR"] - c_row["crude_RR"]) / c_row["crude_RR"] * 100
    print(f"  {var:25s}  crude_RR={c_row['crude_RR']:.3f}  "
          f"adj_RR={a_row['adj_RR']:.3f}  ({change:+.1f}%)")
print("\n→ 變化最大的變項 = 受其他因子干擾最多的因子")

## 題目 3（挑戰題）：模型比較 + Forest Plot

1. 建立兩個 Modified Poisson 模型（精簡 vs 完整）
2. 比較 AIC
3. 用較好的模型畫 Adjusted RR 森林圖

In [ ]:
# --- 模型 A（精簡）：3 個預測因子 ---
model_a = smf.glm(
    "dead ~ age + immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- 模型 B（完整）：5 個預測因子 ---
model_b = smf.glm(
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score",
    data=cases, family=sm.families.Poisson()
).fit(cov_type="HC0", disp=0)

# --- AIC 比較 ---
print("=== 模型比較（Modified Poisson）===")
print(f"  模型 A（3 變項）AIC = {model_a.aic:.1f}")
print(f"  模型 B（5 變項）AIC = {model_b.aic:.1f}")

best = model_a if model_a.aic < model_b.aic else model_b
best_name = "A" if model_a.aic < model_b.aic else "B"
print(f"  → 模型 {best_name} 較佳（AIC 較小 = 解釋力與簡約的最佳平衡）")

In [ ]:
# --- Forest Plot：Adjusted RR（用較好的模型）---
forest_data = []
for var in best.params.index:
    if var == "Intercept":
        continue
    rr = np.exp(best.params[var])
    ci = np.exp(best.conf_int().loc[var])
    forest_data.append({
        "variable": var,
        "RR": rr,
        "ci_lo": ci[0],
        "ci_hi": ci[1],
    })

fdf = pd.DataFrame(forest_data)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(fdf))

# 點估計 + 信賴區間
ax.errorbar(
    fdf["RR"], y_pos,
    xerr=[fdf["RR"] - fdf["ci_lo"], fdf["ci_hi"] - fdf["RR"]],
    fmt="o", color="#D97757", capsize=4, markersize=8,
    ecolor="#6A9BCC", elinewidth=2,
)

# RR = 1 參考線（無效應）
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5, label="RR = 1")

ax.set_yticks(list(y_pos))
ax.set_yticklabels(fdf["variable"])
ax.set_xlabel("Adjusted Risk Ratio (RR)")
ax.set_title(f"死亡預測模型 {best_name} — Adjusted RR 森林圖")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# --- 解讀 ---
print("\n=== 獨立預測因子（RR > 1 且 CI 不包含 1）===")
for _, row in fdf.iterrows():
    sig = "✓ 顯著" if row["ci_lo"] > 1 else "  不顯著"
    print(f"  {row['variable']:25s}  RR={row['RR']:.3f}  "
          f"({row['ci_lo']:.3f}\u2013{row['ci_hi']:.3f})  {sig}")

### 解讀

- **severity_score**：臨床嚴重度是死亡最強的預測因子（RR 最大），這符合直覺
- **immunosuppressed**：控制嚴重度後，免疫抑制可能仍為獨立危險因子
- **age**：年齡每增加一歲的 RR 看起來接近 1，但累積效應大（例如 80 歲 vs 70 歲差 10 歲）
- **RR vs OR**：死亡率 ~16%，OR 和 RR 差距比感染預測（侵襲率 43%）時小，驗證了「盛行率越低，OR 越接近 RR」的原則
- **模型選擇**：AIC 較小的模型不一定每個變項都顯著，但整體平衡較好
- **限制**：死亡人數只有 ~19 人，模型自由度有限，不宜放太多變項